<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks%20/%2004_diavigil_algorithmic_fairness_and_subgroup_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment Setup & Core Deserialization**

In [4]:
# 1. Install missing dependencies in this runtime
!pip install -q catboost

from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np
import re
import catboost
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import CalibratedClassifierCV

# 2. Mount Google Drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# 3. Re-declare custom transformer classes in __main__ for joblib unpickling
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|42613|4262|4263|4264|42651|42652|42653|4266|4267|4268|4270|4272|4273|4276|4278|4279|7850|V450|V533)',
    'elix_valvular_disease': r'^(0932|394|395|396|397|424|7463|7464|7465|7466|V422|V433)',
    'elix_pulmonary_circulation': r'^(416|4179)',
    'elix_peripheral_vascular': r'^(440|441|442|443|444|4471|5571|5579|V434)',
    'elix_hypertension_uncomplicated': r'^(401)',
    'elix_hypertension_complicated': r'^(402|403|404|405)',
    'elix_diabetes_complicated': r'^(250[4-9])',
    'elix_renal_failure': r'^(585|586|V420|V451|V56)',
    'elix_fluid_electrolyte': r'^(276)',
    'elix_chronic_pulmonary': r'^(490|491|492|493|494|495|496|500|501|502|503|504|505)',
    'elix_coagulopathy': r'^(286|2871|2873|2874|2875)'
}

def extract_elixhauser_flags(df: pd.DataFrame) -> pd.DataFrame:
    diag_cols = ['diag_1', 'diag_2', 'diag_3']
    comorb_features = pd.DataFrame(index=df.index)
    cleaned_diags = df[diag_cols].fillna('').astype(str).apply(
        lambda col: col.str.strip().str.replace('.', '', regex=False)
    )
    for flag_name, pattern in ELIXHAUSER_ICD9_RULES.items():
        compiled_regex = re.compile(pattern)
        matches = cleaned_diags.apply(
            lambda row: any(bool(compiled_regex.match(val)) for val in row if val),
            axis=1
        )
        comorb_features[flag_name] = matches.astype(int)
    return comorb_features

class ClinicalDomainFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.medication_columns = [
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
            'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
            'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone'
        ]
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_out = X.copy()
        comorb_df = extract_elixhauser_flags(X_out)
        for col in comorb_df.columns:
            X_out[col] = comorb_df[col].values
        X_out['total_elixhauser_burden'] = comorb_df.sum(axis=1).values
        X_out['healthcare_utilization_index'] = (
            X_out['number_outpatient'].astype(float) +
            (2.0 * X_out['number_emergency'].astype(float)) +
            (3.0 * X_out['number_inpatient'].astype(float))
        )
        active_meds = X_out[self.medication_columns].apply(lambda row: (row != 'No').sum(), axis=1)
        med_titrations = X_out[self.medication_columns].apply(lambda row: row.isin(['Up', 'Down']).sum(), axis=1)
        X_out['active_diabetes_med_count'] = active_meds
        X_out['dosage_titration_count'] = med_titrations
        X_out['on_insulin'] = (X_out['insulin'] != 'No').astype(int)
        X_out['encounter_severity_index'] = (
            X_out['time_in_hospital'].astype(float) *
            (X_out['num_medications'].astype(float) / (X_out['num_lab_procedures'].astype(float) + 1.0))
        )
        X_out['glycemic_status_code'] = X_out['A1Cresult'].map({
            'None': 0, 'Norm': 1, '>7': 2, '>8': 2
        }).fillna(0).astype(int)
        return X_out

# 4. Load the untouched audit test cohort and serialized artifacts
test_parquet_path = os.path.join(DATA_PROCESSED, 'audit_test_cohort.parquet')
artifacts_path = os.path.join(DATA_PROCESSED, 'model_calibration_artifacts.joblib')

assert os.path.exists(test_parquet_path), f"File not found: {test_parquet_path}"
assert os.path.exists(artifacts_path), f"File not found: {artifacts_path}"

test_df = pd.read_parquet(test_parquet_path)
artifacts = joblib.load(artifacts_path)

pipeline = artifacts['fitted_pipeline']
calibrator = artifacts['calibrator']
optimal_threshold = artifacts['optimal_threshold']

print(f"Loaded Untouched Audit Test Cohort: {len(test_df):,} encounters")
print(f"Loaded Optimal Threshold (tau*): {optimal_threshold:.4f}")
print("Cell 1 successfully initialized and artifacts deserialized.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded Untouched Audit Test Cohort: 19,875 encounters
Loaded Optimal Threshold (tau*): 0.1620
Cell 1 successfully initialized and artifacts deserialized.


**Cell 2: Untouched Holdout Test Set Inference**

In [5]:
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score, brier_score_loss

# Feature transformation strictly using pre-fitted pipeline
X_test_clean = test_df.drop(columns=['target', 'encounter_id', 'patient_nbr'])
X_test_trans = pipeline.transform(X_test_clean)
y_test = test_df['target'].values

# Calibrated probability inference
test_cal_probs = calibrator.predict_proba(X_test_trans)[:, 1]
test_preds = (test_cal_probs >= optimal_threshold).astype(int)

print("=== OUT-OF-SAMPLE BENCHMARK (UNTOUCHED TEST SET) ===")
print(f"PR-AUC (Average Precision): {average_precision_score(y_test, test_cal_probs):.4f}")
print(f"ROC-AUC:                   {roc_auc_score(y_test, test_cal_probs):.4f}")
print(f"Brier Score:               {brier_score_loss(y_test, test_cal_probs):.4f}")
print("\nClassification Report at tau* = {:.4f}:".format(optimal_threshold))
print(classification_report(y_test, test_preds, digits=4))

=== OUT-OF-SAMPLE BENCHMARK (UNTOUCHED TEST SET) ===
PR-AUC (Average Precision): 0.2036
ROC-AUC:                   0.6641
Brier Score:               0.0962

Classification Report at tau* = 0.1620:
              precision    recall  f1-score   support

           0     0.9074    0.8815    0.8943     17635
           1     0.2381    0.2915    0.2621      2240

    accuracy                         0.8150     19875
   macro avg     0.5728    0.5865    0.5782     19875
weighted avg     0.8319    0.8150    0.8230     19875



**Cell 3: Algorithmic Fairness Audit Function**

In [6]:
def compute_subgroup_fairness(df, y_true, y_pred, y_prob, group_col):
    results = []
    groups = df[group_col].dropna().unique()

    for g in sorted(groups):
        mask = (df[group_col] == g).values
        n_group = np.sum(mask)
        n_pos = np.sum(y_true[mask] == 1)

        if n_group == 0 or n_pos == 0:
            continue

        tp = np.sum((y_pred[mask] == 1) & (y_true[mask] == 1))
        fn = np.sum((y_pred[mask] == 0) & (y_true[mask] == 1))
        fp = np.sum((y_pred[mask] == 1) & (y_true[mask] == 0))
        tn = np.sum((y_pred[mask] == 0) & (y_true[mask] == 0))

        tpr = tp / (tp + fn)       # Sensitivity / Recall
        fnr = fn / (tp + fn)       # False Negative Rate (Miss Rate)
        fpr = fp / (fp + tn)       # False Positive Rate
        selection_rate = (tp + fp) / n_group
        avg_cal_risk = np.mean(y_prob[mask])

        results.append({
            'Subgroup': str(g),
            'Encounters': int(n_group),
            'Readmitted (<30d)': int(n_pos),
            'Base Prevalence': round(float(n_pos / n_group) * 100, 2),
            'Mean Pred Risk': round(float(avg_cal_risk) * 100, 2),
            'Selection Rate %': round(float(selection_rate) * 100, 2),
            'Sensitivity (TPR)': round(float(tpr), 4),
            'False Neg Rate (FNR)': round(float(fnr), 4),
            'False Pos Rate (FPR)': round(float(fpr), 4)
        })

    return pd.DataFrame(results)

**Cell 4: Fairness Audits Across Age, Gender, and Race**

In [7]:
print("--- AGE GROUP DISPARITY AUDIT ---")
age_fairness_df = compute_subgroup_fairness(test_df, y_test, test_preds, test_cal_probs, 'age')
display(age_fairness_df)

print("\n--- GENDER DISPARITY AUDIT ---")
gender_fairness_df = compute_subgroup_fairness(test_df, y_test, test_preds, test_cal_probs, 'gender')
display(gender_fairness_df)

print("\n--- RACE / ETHNICITY DISPARITY AUDIT ---")
race_fairness_df = compute_subgroup_fairness(test_df, y_test, test_preds, test_cal_probs, 'race')
display(race_fairness_df)

--- AGE GROUP DISPARITY AUDIT ---


,Subgroup,Encounters,Readmitted (<30d),Base Prevalence,Mean Pred Risk,Selection Rate %,Sensitivity (TPR),False Neg Rate (FNR),False Pos Rate (FPR)
0,[0-10),29,1,3.45,2.70,0.00,0.0000,1.0000,0.0000
1,[10-20),128,11,8.59,6.63,6.25,0.0909,0.9091,0.0598
2,[20-30),307,47,15.31,10.31,13.36,0.3404,0.6596,0.0962
3,[30-40),767,89,11.60,11.64,20.47,0.5056,0.4944,0.1652
4,[40-50),1893,191,10.09,11.21,14.47,0.3455,0.6545,0.1222
5,[50-60),3431,341,9.94,10.08,11.13,0.3431,0.6569,0.0858
6,[60-70),4433,486,10.96,11.09,13.04,0.2654,0.7346,0.1138
7,[70-80),5041,603,11.96,12.19,14.90,0.2869,0.7131,0.1302
8,[80-90),3328,413,12.41,12.27,15.29,0.2373,0.7627,0.1410
9,[90-100),518,58,11.20,11.75,8.11,0.1379,0.8621,0.0739



--- GENDER DISPARITY AUDIT ---


,Subgroup,Encounters,Readmitted (<30d),Base Prevalence,Mean Pred Risk,Selection Rate %,Sensitivity (TPR),False Neg Rate (FNR),False Pos Rate (FPR)
0,Female,10792,1188,11.01,11.45,14.02,0.2955,0.7045,0.1210
1,Male,9083,1052,11.58,11.32,13.53,0.2871,0.7129,0.1154



--- RACE / ETHNICITY DISPARITY AUDIT ---


,Subgroup,Encounters,Readmitted (<30d),Base Prevalence,Mean Pred Risk,Selection Rate %,Sensitivity (TPR),False Neg Rate (FNR),False Pos Rate (FPR)
0,?,418,24,5.74,8.35,5.02,0.0833,0.9167,0.0482
1,AfricanAmerican,3787,417,11.01,11.68,15.69,0.3022,0.6978,0.1389
2,Asian,145,13,8.97,10.13,11.03,0.3846,0.6154,0.0833
3,Caucasian,14858,1731,11.65,11.47,13.64,0.2865,0.7135,0.1166
4,Hispanic,376,33,8.78,10.39,13.83,0.4545,0.5455,0.1079
5,Other,291,22,7.56,9.67,11.00,0.4091,0.5909,0.0855


**Cell 5: Equalized Odds & Disparate Impact Disparity Metrics**

Calculate quantitative fairness metrics: Disparate Impact ratio and Maximum False Negative Rate Gap across groups.

In [8]:
# 1. Disparate Impact Ratio across Age: (Min Selection Rate / Max Selection Rate)
min_sel_age = age_fairness_df['Selection Rate %'].min()
max_sel_age = age_fairness_df['Selection Rate %'].max()
di_ratio_age = min_sel_age / max_sel_age

# 2. Maximum FNR Disparity (Clinical Risk Disparity)
max_fnr_age = age_fairness_df['False Neg Rate (FNR)'].max()
min_fnr_age = age_fairness_df['False Neg Rate (FNR)'].min()
fnr_gap_age = max_fnr_age - min_fnr_age

print("=== QUANTITATIVE FAIRNESS AUDIT SUMMARY ===")
print(f"Age Disparate Impact Ratio: {di_ratio_age:.4f} (Benchmark: >0.80 for 80% Rule)")
print(f"Age False Negative Rate Gap: {fnr_gap_age:.4f} (Max FNR: {max_fnr_age:.4f} vs Min FNR: {min_fnr_age:.4f})")

# Save fairness audit summary to disk for paper/presentation tables
fairness_tables = {
    'age_audit': age_fairness_df,
    'gender_audit': gender_fairness_df,
    'race_audit': race_fairness_df,
    'test_cal_probs': test_cal_probs,
    'test_preds': test_preds,
    'y_test': y_test
}
audit_export_path = os.path.join(DATA_PROCESSED, 'fairness_audit_results.joblib')
joblib.dump(fairness_tables, audit_export_path)
print(f"Audit results persisted to: {audit_export_path}")

=== QUANTITATIVE FAIRNESS AUDIT SUMMARY ===
Age Disparate Impact Ratio: 0.0000 (Benchmark: >0.80 for 80% Rule)
Age False Negative Rate Gap: 0.5056 (Max FNR: 1.0000 vs Min FNR: 0.4944)
Audit results persisted to: /content/drive/MyDrive/DiaVigil/data/processed/fairness_audit_results.joblib
